In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [48]:
df = pd.read_csv('marketing_AB.csv')
df.head(5)

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14


A/B Test

We conduct a one-sided A/B test to compare the conversion rates between the treatment group (ad) and the control group (psa).

H0: The conversion rates are the same

H1: The conversion rate for the ad group is higher

Suppose we take significant level alpha = 0.05, power 1-beta = 0.9, 

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

# aggregate
summary = df.groupby('test group')['converted'].agg(
    users='count',
    conversions='sum',
    conversion_rate='mean'
)

print(summary)

             users  conversions  conversion_rate
test group                                      
ad          564577        14423         0.025547
psa          23524          420         0.017854


In [20]:
p_psa = summary.loc['psa', 'conversion_rate']      # psa conversion
p_ad = summary.loc['ad', 'conversion_rate']

print(f"Ad conversion rate:      {p_ad:.4%}")
print(f"PSA conversion rate:     {p_psa:.4%}")
print(f"Observed absolute lift:  {p_ad - p_psa:.4%}")
print(f"Observed relative lift:  {(p_ad - p_psa) / p_psa:.4%}")

Ad conversion rate:      2.5547%
PSA conversion rate:     1.7854%
Observed absolute lift:  0.7692%
Observed relative lift:  43.0851%


The ad treatment increases conversion by 43% (relative lift), which translates to approximately 769 additional conversions per 10,000 users. 

In [ ]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

p_ad_expected = p_psa * 1.5      # expected ad conversion

effect_size = proportion_effectsize(p_ad_expected, p_psa)

analysis = NormalIndPower()
n_per_group = analysis.solve_power(
    effect_size=effect_size,
    alpha=0.05,
    power=0.9,
    alternative='larger'
)

print("Sample size per group:", int(n_per_group))
print("Total sample size:", int(2 * n_per_group))

Sample size per group: 4642
Total sample size: 9285


Assuming a significance level of 0.05 and a desired power of 0.9, the required sample size depends on the expected effect size. For example, detecting a 50% relative increase in conversion requires approximately 9,285 users in total. Since our sample size is larger than this threshold, the experiment appears to be adequately powered for detecting an effect of this magnitude.

In [10]:
# make sure order is [ad, psa]
counts = summary.loc[['ad', 'psa'], 'conversions'].values
nobs = summary.loc[['ad', 'psa'], 'users'].values

# one-sided test (ad > psa)
z_stat, p_value = proportions_ztest(
    count=counts,
    nobs=nobs,
    alternative='larger'
)

print("z-stat:", z_stat)
print("p-value:", p_value)

z-stat: 7.3700781265454145
p-value: 8.526403580779863e-14


The test yields a z-statistic of 7.37 and a p-value of 8.5e−14. Since the p-value is far below the 0.05 significance level, we reject the null hypothesis. This provides strong statistical evidence that the ad treatment leads to a higher conversion rate.

Regression Analysis

To further understand the drivers of conversion, we perform a logistic regression using the features:

total ads (exposure intensity)

most ads day (day of peak exposure)

most ads hour (time of peak exposure)

The results indicate that both the intensity and timing of ad exposure significantly affect conversion. Higher ad exposure is associated with increased conversion odds, while exposure during certain days (e.g., early weekdays) and times (e.g., afternoon and evening) is more effective than late-night or early-morning exposure.

In [11]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_high = confint_proportions_2indep(
    count1 = counts[0], nobs1 = nobs[0], 
    count2 = counts[1], nobs2 = nobs[1], 
    method = 'wald',
    compare = 'diff',
    alpha = 0.05
)

print(f"Ad conversion rate:  {p_ad:.4%}")
print(f"PSA conversion rate: {p_psa:.4%}")
print(f"Absolute lift:       {p_ad - p_psa:.4%}")
print(f"95% CI for lift:     ({ci_low:.4%}, {ci_high:.4%})")

Ad conversion rate:  2.5547%
PSA conversion rate: 1.7854%
Absolute lift:       0.7692%
95% CI for lift:     (0.5951%, 0.9434%)


We are 95% confident that the true lift falls in (0.5951%, 0.9434%). 

In [17]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

n_ad = nobs[0]
n_psa = nobs[1]

ratio = n_ad / n_psa

alpha = 0.05
power_target = 0.9

power_analysis = NormalIndPower()

# -------------------------------------------------
# 1) Minimum Detectable Effect (MDE)
# -------------------------------------------------
# Solve for the smallest detectable treatment conversion rate p1
# given alpha, target power, and current sample sizes.

def achieved_power_for_p(p):
    effect_size = proportion_effectsize(p, p_psa)
    return power_analysis.power(
        effect_size=effect_size,
        nobs1=n_psa,
        alpha=alpha,
        ratio=ratio,
        alternative='larger'
    )

# binary search for p1 > p0 such that achieved power = power_target
low, high = p_psa, 0.999999

for _ in range(100):
    mid = (low + high) / 2
    if achieved_power_for_p(mid) < power_target:
        low = mid
    else:
        high = mid

p_mde = high
mde_abs = p_mde - p_psa
mde_rel = mde_abs / p_psa

print("=== Minimum Detectable Effect (MDE) ===")
print(f"Baseline conversion rate (control): {p_psa:.4%}")
print(f"MDE (absolute lift):               {mde_abs:.4%}")
print(f"MDE (relative lift):               {mde_rel:.2%}")
print(f"Detectable treatment rate:         {p_mde:.4%}")

# -------------------------------------------------
# 2) Achieved power for observed effect
# -------------------------------------------------
effect_size_obs = proportion_effectsize(p_ad, p_psa)

achieved_power = power_analysis.power(
    effect_size=effect_size_obs,
    nobs1=n_psa,
    alpha=alpha,
    ratio=ratio,
    alternative='larger'
)

print("\n=== Achieved Power for Observed Effect ===")
print(f"Observed treatment rate:           {p_ad:.4%}")
print(f"Observed absolute lift:            {p_ad - p_psa:.4%}")
print(f"Observed relative lift:            {(p_ad - p_psa) / p_psa:.2%}")
print(f"Achieved power:                    {achieved_power:.4f}")

=== Minimum Detectable Effect (MDE) ===
Baseline conversion rate (control): 1.7854%
MDE (absolute lift):               0.2670%
MDE (relative lift):               14.95%
Detectable treatment rate:         2.0524%

=== Achieved Power for Observed Effect ===
Observed treatment rate:           2.5547%
Observed absolute lift:            0.7692%
Observed relative lift:            43.09%
Achieved power:                    1.0000


MDE is the mininum detectable effect give the sample size and target alpha and beta. If the observed lift >> MDE, the result is well-powered and stable; if the observed lift <= MDE, the inference is more sensitive to sampling variability and less reliable. 

In [47]:
hetero_day_pre = df.groupby(['most ads day', 'test group'])['converted'].agg(
    count='sum',
    nobs='count',
    conversion_rate = 'mean'
).reset_index()

#print(hetero_day)

results = []

for day in hetero_day_pre['most ads day'].unique():
    day_wise_data = hetero_day_pre[hetero_day_pre['most ads day'] == day]
    #print(day_wise_data)
    
    c_ad = day_wise_data[day_wise_data['test group'] == 'ad']['count'].values.item()
    c_psa = day_wise_data[day_wise_data['test group'] == 'psa']['count'].values.item()

    n_ad = day_wise_data[day_wise_data['test group'] == 'ad'][['nobs']].values.item()
    n_psa = day_wise_data[day_wise_data['test group'] == 'psa'][['nobs']].values.item()

    p_ad = day_wise_data[day_wise_data['test group'] == 'ad'][['conversion_rate']].values.item()
    p_psa = day_wise_data[day_wise_data['test group'] == 'psa'][['conversion_rate']].values.item()

    #print(c_ad, n_ad)
    lift = p_ad - p_psa
    ci_low, ci_high = confint_proportions_2indep(
        count1=c_ad, nobs1=n_ad,
        count2=c_psa, nobs2=n_psa,
        method='wald'
    )

    results.append({
        'day': day,
        'p_ad': p_ad,
        'p_psa': p_psa,
        'lift': lift,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'n_ad': n_ad,
        'n_psa': n_psa})

hetero_day = pd.DataFrame(results).sort_values('lift', ascending= False)
print(hetero_day)

         day      p_ad     p_psa      lift    ci_low   ci_high   n_ad  n_psa
5    Tuesday  0.030440  0.014448  0.015992  0.011483  0.020502  74572   2907
1     Monday  0.033241  0.022559  0.010683  0.005617  0.015749  83571   3502
6  Wednesday  0.025356  0.015759  0.009597  0.005319  0.013874  77418   3490
2   Saturday  0.021307  0.013996  0.007311  0.002888  0.011734  78802   2858
0     Friday  0.022465  0.016303  0.006162  0.002021  0.010303  88805   3803
3     Sunday  0.024620  0.020595  0.004025 -0.001118  0.009168  82332   3059
4   Thursday  0.021637  0.020230  0.001407 -0.003124  0.005937  79077   3905


The treatment effect is positive and statistically significant on most days, with larger lifts observed early in the week (e.g., Monday–Wednesday). For Sunday and Thursday, the confidence intervals include zero, indicating that the effect is not statistically distinguishable from zero, likely due to higher uncertainty in these segments. 

This suggests potential heterogeneity. 

In [53]:
hetero_hour_pre = df.groupby(['most ads hour', 'test group'])['converted'].agg(
    count='sum',
    nobs='count',
    conversion_rate = 'mean'
).reset_index()

#print(hetero_day)

results = []

for hour in hetero_hour_pre['most ads hour'].unique():
    hour_wise_data = hetero_hour_pre[hetero_hour_pre['most ads hour'] == hour]
    #print(hour_wise_data)
    
    c_ad = hour_wise_data[hour_wise_data['test group'] == 'ad']['count'].values.item()
    c_psa = hour_wise_data[hour_wise_data['test group'] == 'psa']['count'].values.item()

    n_ad = hour_wise_data[hour_wise_data['test group'] == 'ad'][['nobs']].values.item()
    n_psa = hour_wise_data[hour_wise_data['test group'] == 'psa'][['nobs']].values.item()

    p_ad = hour_wise_data[hour_wise_data['test group'] == 'ad'][['conversion_rate']].values.item()
    p_psa = hour_wise_data[hour_wise_data['test group'] == 'psa'][['conversion_rate']].values.item()

    #print(c_ad, n_ad)
    lift = p_ad - p_psa
    ci_low, ci_high = confint_proportions_2indep(
        count1=c_ad, nobs1=n_ad,
        count2=c_psa, nobs2=n_psa,
        method='wald'
    )

    results.append({
        'hour': hour,
        'p_ad': p_ad,
        'p_psa': p_psa,
        'lift': lift,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'n_ad': n_ad,
        'n_psa': n_psa})

hetero_hour = pd.DataFrame(results).sort_values('lift', ascending= False)
print(hetero_hour)

    hour      p_ad     p_psa      lift    ci_low   ci_high   n_ad  n_psa
6      6  0.023174  0.000000  0.023174  0.016555  0.029793   1985     83
5      5  0.021563  0.000000  0.021563  0.011112  0.032015    742     23
0      0  0.019213  0.000000  0.019213  0.015520  0.022905   5309    227
4      4  0.015850  0.000000  0.015850  0.006558  0.025142    694     28
1      1  0.013434  0.000000  0.013434  0.010113  0.016756   4615    187
20    20  0.030274  0.017642  0.012632  0.004516  0.020748  27846   1077
14    14  0.028575  0.016051  0.012524  0.006617  0.018431  43779   1869
22    22  0.026455  0.016358  0.010097  0.001654  0.018540  25515    917
23    23  0.022970  0.012924  0.010046  0.000904  0.019188  19547    619
7      7  0.018482  0.008439  0.010044 -0.002078  0.022165   6168    237
8      8  0.019861  0.010622  0.009239  0.001135  0.017342  16968    659
9      9  0.019529  0.010815  0.008714  0.002659  0.014768  29802   1202
13    13  0.025063  0.016590  0.008473  0.002911  0

Early-hour segments show large estimated lifts, but PSA sample sizes are very small, leading to unstable estimates and zero observed conversions. 

Among well-sampled hours, the estimated treatment effects are relatively consistent with overlapping confidence intervals, suggesting limited evidence of strong heterogeneity across hours

In [55]:
df_ad_group = df[df['test group'] == 'ad']
# --------- features ----------
X = df_ad_group[['total ads', 'most ads day', 'most ads hour']]
y = df_ad_group['converted']

# --------- column types ----------
num_cols = ['total ads']
cat_cols = ['most ads day', 'most ads hour']

# --------- preprocessing ----------
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop=[['Friday'], [0]]), cat_cols)
    ]
)

# --------- model ----------
model = LogisticRegression(max_iter=1000)

# --------- pipeline ----------
pipe = Pipeline([
    ('preprocess', preprocess),
    ('model', model)
])

# --------- split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --------- fit ----------
pipe.fit(X_train, y_train)

# --------- score ----------
print("Train score:", pipe.score(X_train, y_train))
print("Test score:", pipe.score(X_test, y_test))

Train score: 0.97332955468814
Test score: 0.9726788054837224


In [23]:
feature_names = (
    num_cols +
    list(pipe.named_steps['preprocess']
         .named_transformers_['cat']
         .get_feature_names_out(cat_cols))
)

coefs = pipe.named_steps['model'].coef_[0]

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': coefs
}).sort_values(by='coef', ascending=False)

print(coef_df)

                   feature      coef
5     most ads day_Tuesday  0.528934
1      most ads day_Monday  0.508023
0                total ads  0.449273
6   most ads day_Wednesday  0.319366
3      most ads day_Sunday  0.307730
22        most ads hour_16  0.298055
20        most ads hour_14  0.163263
26        most ads hour_20  0.156628
27        most ads hour_21  0.138675
21        most ads hour_15  0.131329
4    most ads day_Thursday  0.122775
23        most ads hour_17  0.099473
28        most ads hour_22  0.098202
2    most ads day_Saturday  0.088061
29        most ads hour_23  0.069332
24        most ads hour_18  0.044360
25        most ads hour_19  0.033749
19        most ads hour_13 -0.031616
18        most ads hour_12 -0.127626
16        most ads hour_10 -0.182206
10         most ads hour_4 -0.188403
11         most ads hour_5 -0.203184
17        most ads hour_11 -0.210467
15         most ads hour_9 -0.224034
14         most ads hour_8 -0.342153
12         most ads hour_6 -0.345933
9

In [24]:
y_pred = pipe.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_pred)

np.float64(0.8102428896660834)

The coefficient for total ads is positive (0.449), indicating that higher ad exposure is associated with higher odds of conversion. Specifically, each additional ad increases the odds of conversion by approximately 57%, holding other factors constant.

The timing of ad exposure also plays an important role. Users whose peak exposure occurs between Sunday and Wednesday tend to have higher conversion odds, while exposure during midnight and early morning hours is associated with lower conversion odds.

All categorical effects are interpreted relative to the baseline categories (Friday and 12am). A positive coefficient indicates higher conversion odds compared to the baseline, while a negative coefficient indicates lower conversion odds.

In [65]:
df_test = X_test.copy()
df_test['y_true'] = y_test.values
#y_pred = pipe.predict_proba(X_test)[:, 1]
df_test['score'] = y_pred

# create deciles
df_test['ventiles'] = pd.qcut(df_test['score'], 20, labels=False)

# evaluate conversion rate per decile
ventile_perf = df_test.groupby('ventiles')['y_true'].mean()

print(ventile_perf.sort_index(ascending=False))

ventiles
19    0.181931
18    0.095356
17    0.048750
16    0.029735
15    0.021597
14    0.017715
13    0.019371
12    0.018031
11    0.011858
10    0.012040
9     0.007814
8     0.007592
7     0.010508
6     0.007835
5     0.007732
4     0.006008
3     0.004440
2     0.006008
1     0.004965
0     0.003712
Name: y_true, dtype: float64


We segment users into score-based ventiles using predicted conversion probabilities. This allows us to evaluate ranking performance by comparing observed conversion rates across quantile-based groups. 

The model demonstrates strong ranking ability, with conversion rates increasing substantially in higher score quantiles. In particular, the top 15% of users exhibit significantly higher conversion rates (approximately 5%–18%) compared to the overall baseline (~2%). This suggests that prioritizing treatment for high-propensity users can substantially improve targeting efficiency.